In [1]:
from dj_notebook import activate

plus = activate()

Output()

In [2]:
import json
from simple_history.utils import update_change_reason

def registrar_movimiento_stock(stock_item, cantidad_delta, motivo, usuario=None, contexto=None):
    """
    Registra un cambio de stock con motivo y contexto adicional.
    
    Args:
        stock_item (StockItemEnBodega): el stock que se va a actualizar
        cantidad_delta (int): el cambio de cantidad (+entrada, -salida)
        motivo (str): descripción del movimiento
        usuario (User, optional): usuario que hace el cambio
        contexto (dict, optional): datos adicionales como {"origen": "OC-123"}
    """
    if not hasattr(stock_item, 'save'):
        raise ValueError("El objeto debe ser un modelo Django válido con historial")

    # Actualizar stock
    stock_item.cantidad += cantidad_delta

    # Preparar razón de cambio
    descripcion = {
        "motivo": motivo,
        "cantidad_delta": cantidad_delta,
        "nuevo_stock": stock_item.cantidad,
        "contexto": contexto or {}
    }

    # Registrar el motivo de cambio
    razon = f"{motivo} | {json.dumps(descripcion, ensure_ascii=False)}"
    update_change_reason(stock_item, razon)

    # Asociar usuario si se provee
    if usuario:
        stock_item._history_user = usuario

    # Guardar cambios
    stock_item.save(update_fields=["cantidad"])


In [9]:
from bodegas.models import StockItemEnBodega
from cuentas.models import User

In [6]:
for x in StockItemEnBodega.objects.all():
    print(x.cantidad, x)

1 Monster sin azucar en Insumos varios


In [7]:
item = StockItemEnBodega.objects.get(pk=1)

In [11]:
motivo = "devolucion de testing 001"
usuario = User.objects.get(pk=1)
contexto = {"guia_salida_id": 1,"item_guia_id": 1}
registrar_movimiento_stock(item, cantidad_delta=1, motivo=motivo, usuario=usuario, contexto=contexto)

AttributeError: 'NoneType' object has no attribute 'history_change_reason'